# Trening modela — LSTM vs. BERT

U ovom notebooku treniramo oba modela za klasifikaciju support tiketa u **10 kategorija**.

**Hiperparametri:**

| Model | Learning rate | Batch size | Epochs |
|-------|---------------|------------|--------|
| LSTM  | 0.001         | 32         | 20     |
| BERT  | 2e-5          | 16         | 5      |

Oba modela koriste **early stopping** (patience=3), **ReduceLROnPlateau** scheduler i čuvaju najbolji checkpoint u `models/` folderu.

> **Napomena:** BERT trening može trajati dugo na CPU. Preporučuje se GPU/MPS ako je dostupan.

## 1. Priprema okruženja

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.train import (
    get_device,
    load_processed_dataloaders,
    plot_training_curves,
    set_seed,
    train_bert,
    train_lstm,
)

set_seed()
device = get_device()
print(f"Uređaj: {device}")

## 2. Pregled podataka

In [ ]:
train_loader, val_loader, test_loader = load_processed_dataloaders(config.LSTM_TRAIN_BATCH_SIZE)

print(f"Train batches: {len(train_loader)} (batch_size={config.LSTM_TRAIN_BATCH_SIZE})")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Train primera: {len(train_loader.dataset):,}")
print(f"Val primera: {len(val_loader.dataset):,}")
print(f"Test primera: {len(test_loader.dataset):,}")

## 3. Trening LSTM modela

In [ ]:
lstm_trainer, lstm_history = train_lstm(device=device)
lstm_history_df = pd.DataFrame(lstm_history)
lstm_history_df

## 4. Krivulje treninga — LSTM

In [ ]:
plot_training_curves(
    lstm_history,
    model_name="lstm",
    save_path=config.TRAINING_HISTORY_DIR / "lstm_curves.png",
)

## 5. Trening BERT modela

In [ ]:
bert_trainer, bert_history = train_bert(device=device)
bert_history_df = pd.DataFrame(bert_history)
bert_history_df

## 6. Krivulje treninga — BERT

In [ ]:
plot_training_curves(
    bert_history,
    model_name="bert",
    save_path=config.TRAINING_HISTORY_DIR / "bert_curves.png",
)

## 7. Poređenje rezultata

In [ ]:
lstm_best = lstm_history_df.loc[lstm_history_df["val_loss"].idxmin()]
bert_best = bert_history_df.loc[bert_history_df["val_loss"].idxmin()]

comparison = pd.DataFrame([
    {"model": "LSTM", "metric": "accuracy", "value": lstm_best["val_accuracy"]},
    {"model": "LSTM", "metric": "f1", "value": lstm_best["val_f1"]},
    {"model": "BERT", "metric": "accuracy", "value": bert_best["val_accuracy"]},
    {"model": "BERT", "metric": "f1", "value": bert_best["val_f1"]},
])
comparison

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=comparison, x="metric", y="value", hue="model", palette="Set2")
plt.title("Poređenje najboljih validation metrika")
plt.ylabel("Vrednost")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

## Zaključak

Trening je završen za oba modela:
- **LSTM checkpoint:** `models/best_lstm.pt`
- **BERT checkpoint:** `models/best_bert.pt`
- **Istorija treninga:** `logs/training/lstm_history.json` i `logs/training/bert_history.json`

Sledeći korak: evaluacija na test skupu i poređenje performansi modela.